## Three stages similarity calculation

In [1]:
import numpy as np
import pandas as pd


def load_mean_adj(graph_path, label_path):
    graphs = np.load(graph_path)
    adj = graphs.mean(axis=0)
    df = pd.read_csv(label_path)
    labels = df.iloc[:, 0].astype(str).str.strip().values
    return adj, labels


# paths
stage1_graph_path_2 = './result/wo_layer/stage1/tuning/bs32_dp0.1_sw1.0/dynamic_full_graphs.npy'
stage1_label_path_2 = './result/wo_layer/stage1/df_scaled.csv'

stage2_graph_path_2 = './result/wo_layer/stage2/tuning/bs32_dp0.1_sw1.0/dynamic_full_graphs.npy'
stage2_label_path_2 = './result/wo_layer/stage1/df_scaled.csv'

stage3_graph_path_2 = './result/wo_layer/stage3/tuning/bs32_dp0.1_sw0.5/dynamic_full_graphs.npy'
stage3_label_path_2 = './result/wo_layer/stage1/df_scaled.csv'


A1, L1 = load_mean_adj(stage1_graph_path_2, stage1_label_path_2)
A2, L2 = load_mean_adj(stage2_graph_path_2, stage2_label_path_2)
A3, L3 = load_mean_adj(stage3_graph_path_2, stage3_label_path_2)


# find intersection regions
common = sorted(list(set(L1) & set(L2) & set(L3)))


def reorder(adj, labels, common):
    idx = [np.where(labels == c)[0][0] for c in common]
    return adj[np.ix_(idx, idx)]


A1c = reorder(A1, L1, common)
A2c = reorder(A2, L2, common)
A3c = reorder(A3, L3, common)


def edge_mae(A, B, eps=0, mode="all"):
    if A.shape != B.shape:
        raise ValueError("A and B must have the same shape.")

    n = A.shape[0]

    if mode == "offdiag":
        region_mask = ~np.eye(n, dtype=bool)
    elif mode == "diagonal":
        region_mask = np.eye(n, dtype=bool)
    elif mode == "all":
        region_mask = np.ones((n, n), dtype=bool)
    else:
        raise ValueError("mode must be 'offdiag', 'diagonal', or 'all'.")

    valid_mask = region_mask & (
        (np.abs(A) > eps) | (np.abs(B) > eps)
    )

    if valid_mask.sum() == 0:
        return np.nan

    return np.abs(A[valid_mask] - B[valid_mask]).mean()


mae12 = edge_mae(A1c, A2c)
mae13 = edge_mae(A1c, A3c)
mae23 = edge_mae(A2c, A3c)

print("Intersection brain regions:", len(common))
print("Edge MAE (Stage1 vs Stage2):", round(mae12, 4))
print("Edge MAE (Stage1 vs Stage3):", round(mae13, 4))
print("Edge MAE (Stage2 vs Stage3):", round(mae23, 4))

Intersection brain regions: 89
Edge MAE (Stage1 vs Stage2): 0.1306
Edge MAE (Stage1 vs Stage3): 0.2179
Edge MAE (Stage2 vs Stage3): 0.2241


In [2]:
stage_variation = np.mean([mae12, mae13, mae23])

print(f"Stage variation: {stage_variation:.4f}")

Stage variation: 0.1909


In [3]:
def top5_change_contribution(A, B, eps=0):
    mask = ~np.eye(A.shape[0], dtype=bool)
    mask &= (np.abs(A) > eps) | (np.abs(B) > eps)

    changes = np.abs(A[mask] - B[mask])
    n_top = max(1, int(np.ceil(len(changes) * 0.05)))

    return changes.size, n_top, 100 * np.sort(changes)[-n_top:].sum() / changes.sum()


for name, A, B in [
    ("Stage I vs Stage II", A1c, A2c),
    ("Stage I vs Stage III", A1c, A3c),
    ("Stage II vs Stage III", A2c, A3c)
]:
    n_edges, n_top, contribution = top5_change_contribution(A, B)

    print(
        f"{name}: top 5% edges ({n_top}/{n_edges}) "
        f"contribute {contribution:.2f}% of total change"
    )

Stage I vs Stage II: top 5% edges (392/7832) contribute 37.00% of total change
Stage I vs Stage III: top 5% edges (392/7832) contribute 22.71% of total change
Stage II vs Stage III: top 5% edges (392/7832) contribute 22.08% of total change


In [4]:
import numpy as np
import pandas as pd
import mpmath as mp
from scipy.stats import pearsonr

mp.mp.dps = 5000

# ============================================================
# load learned graphs
# ============================================================
def load_mean_adj(graph_path, label_path):
    graphs = np.load(graph_path)
    adj = graphs.mean(axis=0)

    df = pd.read_csv(label_path)
    labels = df.iloc[:, 0].astype(str).str.strip().values

    return adj, labels


# ============================================================
# reorder matrix
# ============================================================
def reorder(adj, labels, common):
    labels = np.asarray(labels).astype(str)
    idx = [np.where(labels == c)[0][0] for c in common]
    return adj[np.ix_(idx, idx)]


# ============================================================
# high-precision Pearson p-value
# ============================================================
def pearson_p_mpmath(r, n, digits=30):
    r = mp.mpf(str(abs(r)))
    n = mp.mpf(str(n))

    df = n - 2

    # two-sided p-value for Pearson correlation
    # p = I_{1-r^2}(df/2, 1/2)
    x = 1 - r**2
    p = mp.betainc(df / 2, mp.mpf("0.5"), 0, x, regularized=True)

    return mp.nstr(p, digits, min_fixed=0, max_fixed=0)


# ============================================================
# compare matrices
# ============================================================
def compare_matrix(A,B):
    A,B=np.asarray(A,float),np.asarray(B,float)
    m=~np.eye(A.shape[0],dtype=bool)
    a,b=np.abs(A[m]),np.abs(B[m])
    v=np.isfinite(a)&np.isfinite(b); a,b=a[v],b[v]
    r,_=pearsonr(a,b)
    return {
        "MAE":np.mean(np.abs(a-b)),
        "RMSE":np.sqrt(np.mean((a-b)**2)),
        "Pearson_r":r,
        "Pearson_p":pearson_p_mpmath(r,len(a),digits=3)
    }

# ============================================================
# learned graphs
# ============================================================
stage1_graph_path_2 = './result/wo_layer/stage1/tuning/bs32_dp0.1_sw1.0/dynamic_full_graphs.npy'
stage1_label_path_2 = './result/wo_layer/stage1/df_scaled.csv'

stage2_graph_path_2 = './result/wo_layer/stage2/tuning/bs32_dp0.1_sw1.0/dynamic_full_graphs.npy'
stage2_label_path_2 = './result/wo_layer/stage1/df_scaled.csv'

stage3_graph_path_2 = './result/wo_layer/stage3/tuning/bs32_dp0.1_sw0.5/dynamic_full_graphs.npy'
stage3_label_path_2 = './result/wo_layer/stage1/df_scaled.csv'

# ============================================================
# load learned graphs
# ============================================================
A1, L1 = load_mean_adj(stage1_graph_path_2, stage1_label_path_2)
A2, L2 = load_mean_adj(stage2_graph_path_2, stage2_label_path_2)
A3, L3 = load_mean_adj(stage3_graph_path_2, stage3_label_path_2)


# ============================================================
# load layer matrix
# ============================================================
df_layers = pd.read_excel(
    '../../SEU_net_wo_corticalLayers.xlsx',
    header=1,
    index_col=1
)

df_layers = df_layers.iloc[:, 1:]

layer_labels = df_layers.index.astype(str).str.strip().values
A_layer = df_layers.values.astype(float)


# ============================================================
# intersection
# ============================================================
common = sorted(list(set(L1) & set(L2) & set(L3) & set(layer_labels)))

print(f"\nIntersection brain regions: {len(common)}")


# ============================================================
# reorder
# ============================================================
A1c = reorder(A1, L1, common)
A2c = reorder(A2, L2, common)
A3c = reorder(A3, L3, common)
Alc = reorder(A_layer, layer_labels, common)


# ============================================================
# compare
# ============================================================
results = {
    "Stage1_vs_Layer": compare_matrix(A1c, Alc),
    "Stage2_vs_Layer": compare_matrix(A2c, Alc),
    "Stage3_vs_Layer": compare_matrix(A3c, Alc),
}

df_results = pd.DataFrame(results).T


# ============================================================
# print only
# ============================================================
df_print = df_results.copy()

df_print["MAE"] = df_print["MAE"].map(lambda x: f"{x:.4f}")
df_print["RMSE"] = df_print["RMSE"].map(lambda x: f"{x:.4f}")
df_print["Pearson_r"] = df_print["Pearson_r"].map(lambda x: f"{x:.4f}")

print("\nComparison results:")
print(df_print)


Intersection brain regions: 89

Comparison results:
                    MAE    RMSE Pearson_r Pearson_p
Stage1_vs_Layer  0.5099  0.7907    0.0027   8.08e-1
Stage2_vs_Layer  0.5267  0.7954    0.0070   5.38e-1
Stage3_vs_Layer  0.5205  0.7943   -0.0047   6.74e-1


In [5]:
mean_pearson_r = df_results["Pearson_r"].astype(float).mean()

print(f"\nMean Pearson correlation: {mean_pearson_r:.4f}")


Mean Pearson correlation: 0.0017


In [6]:
import numpy as np, pandas as pd, mpmath as mp
from scipy.stats import pearsonr

mp.mp.dps = 100

paths = [
    ('./result/wo_layer/stage1/tuning/bs32_dp0.1_sw1.0/dynamic_full_graphs.npy',
     './result/wo_layer/stage1/df_scaled.csv'),
    ('./result/wo_layer/stage2/tuning/bs32_dp0.1_sw1.0/dynamic_full_graphs.npy',
     './result/wo_layer/stage1/df_scaled.csv'),
    ('./result/wo_layer/stage3/tuning/bs32_dp0.1_sw0.5/dynamic_full_graphs.npy',
     './result/wo_layer/stage1/df_scaled.csv')
]

def load(g,l):
    return np.load(g).mean(0), pd.read_csv(l).iloc[:,0].astype(str).str.strip().values

def pval(r,n):
    x=1-mp.mpf(str(abs(r)))**2
    return mp.nstr(mp.betainc((n-2)/2,.5,0,x,regularized=True),3,min_fixed=0,max_fixed=0)

A,L = zip(*[load(*p) for p in paths])
common = sorted(set(L[0]) & set(L[1]) & set(L[2]))

A = [
    a[np.ix_(
        [np.where(l==c)[0][0] for c in common],
        [np.where(l==c)[0][0] for c in common]
    )]
    for a,l in zip(A,L)
]

mask = ~np.eye(len(common),dtype=bool)
pairs = [(0,1),(0,2),(1,2)]
names = ["I–II","I–III","II–III"]

res = []
for (i,j),name in zip(pairs,names):
    x,y=A[i][mask],A[j][mask]
    m=np.isfinite(x)&np.isfinite(y)
    x,y=x[m],y[m]
    r,_=pearsonr(x,y)
    res.append([name,np.mean(abs(x-y)),np.sqrt(np.mean((x-y)**2)),r,pval(r,len(x))])

df = pd.DataFrame(res,columns=["Comparison","MAE","RMSE","Pearson_r","Pearson_p"])
print(df.to_string(index=False,formatters={
    "MAE":"{:.4f}".format,
    "RMSE":"{:.4f}".format,
    "Pearson_r":"{:.4f}".format
}))

r=df["Pearson_r"].to_numpy()
rel=(1-r)/(1-r).sum()

print("\nRelative stage separation:")
for n,x in zip(names,rel):
    print(f"{n}: {x:.1%}")

Comparison    MAE   RMSE Pearson_r  Pearson_p
      I–II 0.1321 0.2964    0.7910 1.49e-1673
     I–III 0.2204 0.4113    0.5922  5.73e-737
    II–III 0.2266 0.4312    0.5670  7.73e-662

Relative stage separation:
I–II: 19.9%
I–III: 38.8%
II–III: 41.3%


## raw function data

In [7]:
import pickle
import numpy as np
from scipy.stats import pearsonr, spearmanr

D = []
for i in range(1, 4):
    with open(f"../result/wo_layer/stage{i}/binned_traces_stage_{i}.pkl", "rb") as f:
        D.append(pickle.load(f))

regions = sorted(set.intersection(*(
    set(d["binned_traces"].keys()) for d in D
)))

profiles = np.array([
    [
        np.nanmean(np.asarray(d["binned_traces"][r], dtype=float))
        for r in regions
    ]
    for d in D
])

for i, j in [(0,1), (0,2), (1,2)]:
    m = np.isfinite(profiles[i]) & np.isfinite(profiles[j])

    pr = pearsonr(profiles[i][m], profiles[j][m])
    sr = spearmanr(profiles[i][m], profiles[j][m])

    print(
        f"Stage {i+1} vs {j+1}: "
        f"Pearson r={pr.statistic:.3f}, p={pr.pvalue:.3e}; "
        f"Spearman rho={sr.statistic:.3f}, p={sr.pvalue:.3e}"
    )

Stage 1 vs 2: Pearson r=0.773, p=7.251e-19; Spearman rho=0.808, p=1.131e-21
Stage 1 vs 3: Pearson r=0.496, p=7.885e-07; Spearman rho=0.415, p=5.349e-05
Stage 2 vs 3: Pearson r=0.353, p=6.820e-04; Spearman rho=0.336, p=1.308e-03


In [8]:
import pickle, numpy as np
from scipy.stats import pearsonr, spearmanr

N=1000
D=[pickle.load(open(f"../result/wo_layer/stage{i}/binned_traces_stage_{i}.pkl","rb")) for i in range(1,4)]

regions=sorted(set.intersection(*(
    set(d["binned_traces"]) for d in D
)))

def trace(d):
    b=d["binned_traces"];X=[]
    for r in regions:
        x=np.asarray(b[r],float)
        edges=np.linspace(0,len(x),N+1).astype(int)
        y=np.array([
            np.nanmean(x[edges[k]:edges[k+1]])
            if edges[k+1]>edges[k] else np.nan
            for k in range(N)
        ])
        X.append(y)
    return np.asarray(X)

X=[trace(d) for d in D]

print("Whole region × time:")
for i,j in [(0,1),(0,2),(1,2)]:
    x,y=X[i].ravel(),X[j].ravel()
    m=np.isfinite(x)&np.isfinite(y)

    print(
        f"{i+1}-{j+1}: "
        f"Pearson={pearsonr(x[m],y[m]).statistic:.3f}, "
        f"Spearman={spearmanr(x[m],y[m]).statistic:.3f}"
    )

Whole region × time:
1-2: Pearson=0.389, Spearman=0.430
1-3: Pearson=0.154, Spearman=0.147
2-3: Pearson=0.208, Spearman=0.189


In [9]:
import numpy as np
from scipy.stats import pearsonr, spearmanr

N_BOOT = 10000
SEED = 42
rng = np.random.default_rng(SEED)

def corr3(x, method="pearson"):
    fun = pearsonr if method == "pearson" else spearmanr
    r12 = fun(x[0], x[1]).statistic
    r13 = fun(x[0], x[2]).statistic
    r23 = fun(x[1], x[2]).statistic
    delta = r23 - (r12 + r13) / 2
    return r12, r13, r23, delta

for method in ["pearson", "spearman"]:

    obs = corr3(profiles, method)

    boots = []
    n = profiles.shape[1]

    for _ in range(N_BOOT):
        idx = rng.integers(0, n, n)
        vals = corr3(profiles[:, idx], method)

        if np.all(np.isfinite(vals)):
            boots.append(vals)

    boots = np.asarray(boots)

    print(f"\n{method.upper()}")
    print("-" * 45)

    names = ["I-II", "I-III", "II-III", "Δ contrast"]

    for k, name in enumerate(names):
        lo, hi = np.percentile(boots[:, k], [2.5, 97.5])
        print(
            f"{name:10s}: {obs[k]:.4f} "
            f"[95% CI {lo:.4f}, {hi:.4f}]"
        )

    # bootstrap probability that Delta <= 0
    p = (np.sum(boots[:, 3] <= 0) + 1) / (len(boots) + 1)

    print(f"P(Δ <= 0) = {p:.6f}")


PEARSON
---------------------------------------------
I-II      : 0.7728 [95% CI 0.6764, 0.8511]
I-III     : 0.4956 [95% CI 0.3116, 0.6421]
II-III    : 0.3534 [95% CI 0.1761, 0.5171]
Δ contrast: -0.2808 [95% CI -0.4250, -0.1278]
P(Δ <= 0) = 0.999900

SPEARMAN
---------------------------------------------
I-II      : 0.8078 [95% CI 0.6996, 0.8766]
I-III     : 0.4146 [95% CI 0.2159, 0.5826]
II-III    : 0.3355 [95% CI 0.1281, 0.5128]
Δ contrast: -0.2757 [95% CI -0.4293, -0.1184]
P(Δ <= 0) = 0.999700
